# Hackathon 1, statistics.

This project illustrates the statistics part of the course LEPL1109. In the first part of the project, you will study the China water pollution by analyzing a dataset providing the water pollution levels collected from various monitoring stations across 10 major provinces in China throughout the year 2023. In the second part of the project, you will analyze a dataset containing high-frequency time-series  data collected from an industrial boiler operating in a chemical plant.

## Report content

•	Grades are granted to the members whose names are in the Jupyter notebook. If your name doesn’t appear on the top of the notebook, you’ll get a 0, even though you are in a group on Moodle.

•	The jupyter notebook must be compiled with printed results and next submitted via moodle. The absence of compiled results (or non-printed values) leads to a lower grade.

•	Do not comment your results directly into cells of code. Use instead a Markdown cell. 

•	"Dry" code or results not followed by a minimum of analysis / comments will be penalized.


## Report submission

•	Deadline, see moodle website. Submission after the deadline will not be accepted.

•	To submit your report, go to the section “APP” on Moodle and the subsection “Soumission du rapport”. You can upload your work there. Once you are sure that it is your final version, click the button “Envoyer le devoir”. It is important that you don’t forget to click on this button ! 

•	Reports that have not been uploaded through Moodle will not be corrected.


## Names and Noma of participants:

Part. 1: Djoukouo Noubissi Marie Pascale

Part. 2: Kryvobokova Kateryna

Part. 3: Pinter Aurélie

Part. 4: Rencelot Maëlle

Part. 5: Tannir Dana

Part. 6: Van Aster Margaux

# China Water Pollution 

This dataset provides  water pollution levels collected from various monitoring stations across 10 major provinces in China throughout the year 2023. The data  includes  parameters such as pH, turbidity, chemical and biological oxygen demand, nutrient levels, and heavy metal concentrations. These indicators are widely used by environmental monitoring agencies to assess water quality for ecological, human, and industrial impacts.

We will focus on the Water Quality Index. 

## 1. Basic statistics

In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st 

import statsmodels.api as sm # pour la regression via OLS()
from sklearn.model_selection import train_test_split # pour le split train/test

from sklearn.gaussian_process import GaussianProcessRegressor # pour la regression via GRP
from sklearn.gaussian_process.kernels import RBF, Matern

1.a) Load the dataset 'china_water_pollution_data_hack.csv'. Convert Province, City  to categorical variables. (**0.5 pt**) 

In [62]:
df = pd.read_csv("china_water_pollution_data_hack.csv", usecols=["Province", "City", "Date", "Water_Quality_Index"], dtype={"Province": "category", "City" : "category", "Water_Quality_Index" : "float"}, parse_dates=["Date"])  

In [63]:
df = pd.read_csv("china_water_pollution_data_hack.csv", dtype={"Province": "category", "City" : "category", "Water_Quality_Index" : "float"}, parse_dates=["Date"])  

1.b) Calculate the mean, variance, median, 25% and 75% quantiles of the water quality index (which ranges from 0 to 100) for all cities in the dataset. Comment your results! (**1.5 pts**)

In [54]:
#code here
print(df["City"].unique())
print(type(df.loc[df["City"] == "Shanghai", "Water_Quality_Index"]))

dict_stat = {}      # Dictionnaire de la forme : {city : [mean, variance, median, 25% quantile, 75% quantile]} pour chaque ville du dataset
for city in df["City"].unique():
    dict_stat[city] = []
    wqi_city = df.loc[df["City"] == city, "Water_Quality_Index"]
    dict_stat[city].append(pd.Series.mean(wqi_city))
    dict_stat[city].append(pd.Series.var(wqi_city))
    dict_stat[city].append(pd.Series.median(wqi_city))
    dict_stat[city].append(wqi_city.quantile(q=0.25))
    dict_stat[city].append(wqi_city.quantile(q=0.75))

['Ningbo', 'Mianyang', 'Beijing', 'Chengdu', 'Dali', ..., 'Zhengzhou', 'Kunming', 'Shanghai', 'Suzhou', 'Wuhan']
Length: 18
Categories (18, object): ['Beijing', 'Chengdu', 'Dali', 'Guangzhou', ..., 'Suzhou', 'Wuhan', 'Yichang', 'Zhengzhou']
<class 'pandas.core.series.Series'>


Comment here:

## 2. Hypothesis tests 

2.a) Check with a Student's T test that the average water quality index is the same in Shenzhen and Dali: $$H_0: \mu_{Shenzhen} = \mu_{Dali},$$ 
$$H_1: \mu_{Shenzhen} \neq \mu_{Dali}.$$ Calculate all statistics and p-value without recourse to other functions than statistical distributions (use course's formula). Use a confidence level of 5%. Take care to comment your conclusions. Are all assumptions required to perform this test sastisfied? Which additional test do you have to do to validate your result? (**2.5 pts**)

In [ ]:
#code here


Comment here:

2.b) 'Wuhan', 'Luoyang', 'Chengdu', 'Nanjing', 'Dali' seems to have similar (and low) water quality index. Test the assumption: $$H_0:  \mu_{Wuhan} = \mu_{Luoyang}= \mu_{Chengdu} = \mu_{Nanjing} =\mu_{Dali}.$$
**Hint**: reformulate the problem as a linear regression.

(**2 pts**)

In [ ]:
#code here


Comment here: 

## 3. Regressions

3.a) Propose a regression model which explains the Water_Quality_Index as a function of other explanatory variables, **for the city of Shanghai**. Split your data set into a training set (80% of the data) that you use for fitting the model and a test set (20% of the data) on which you test the accuracy of the model. 

* Use the OLS() function of the package statsmodels.api to perform the linear regression. 
* Comment your results (goodness of fit, R2, F-stat and T-stats of coefficients)
* Identify potential non-relevant covariates
* Calculate the MAE on the test and training sets. 

(**3 pts**)

In [64]:
df_shanghai = df[df['City'] == 'Shanghai']   # on filtre les données de Shanghai

Y = df_shanghai['Water_Quality_Index']   # variable reponse Yi

X = df_shanghai.drop(columns=['Water_Quality_Index', 'Province', 'City', 'Date'])   # matrice des variables explicatives Xi
X = sm.add_constant(X)   # on y ajoute une colonne de 1 au début de la matrice pour le terme beta_0

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [65]:
# modèle de régression linéaire via OLS()

model = sm.OLS(Y_train, X_train)   # création du modèle
results = model.fit()   # ajustement aux données

print(results.summary()) 

# covariables non pertinentes : si p-value > 0.05 (P>|t| dans le tableau)
# Water_Temperature_C, pH, Conductivity_uS_cm, Nitrite_mg_L et Total_Nitrogen_mg_L

mae_train = np.mean(np.abs(Y_train - results.predict(X_train)))   # calcul du MAE (Mean Absolute Error = epsilon) sur les 2 sets
mae_test = np.mean(np.abs(Y_test - results.predict(X_test)))

print("Epsilon du training set = ", mae_train)
print("Epsilon du test set = ", mae_test)

                             OLS Regression Results                            
Dep. Variable:     Water_Quality_Index   R-squared:                       0.777
Model:                             OLS   Adj. R-squared:                  0.761
Method:                  Least Squares   F-statistic:                     50.46
Date:                 Wed, 15 Oct 2025   Prob (F-statistic):           3.51e-66
Time:                         16:37:09   Log-Likelihood:                -544.84
No. Observations:                  249   AIC:                             1124.
Df Residuals:                      232   BIC:                             1183.
Df Model:                           16                                         
Covariance Type:             nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const         

Comment here: 

3.b) Same question as 3.a) but now you use a Gaussian process regression. Use a RBF and Matern kernel and compare MAEs of the 2 models. Which one is the best? (**2 pts**)

In [67]:
# création du modèle de GRP via les noyaux RBF et Matern 
rbf = RBF(length_scale=1.0) 
gp_rbf = GaussianProcessRegressor(kernel=rbf)

matern = Matern(length_scale=1.0, nu=1.5) 
gp_matern = GaussianProcessRegressor(kernel=matern) 

# ajustement des modèles sur le training set
gp_rbf.fit(X_train, Y_train)
gp_matern.fit(X_train, Y_train)

# calcul du MAE (Mean Absolute Error = epsilon) sur les 2 sets  
mae_train_rbf = np.mean(np.abs(Y_train - gp_rbf.predict(X_train)))
mae_test_rbf = np.mean(np.abs(Y_test - gp_rbf.predict(X_test))) 

mae_train_matern = np.mean(np.abs(Y_train - gp_matern.predict(X_train)))
mae_test_matern = np.mean(np.abs(Y_test - gp_matern.predict(X_test)))

print("Epsilon du training set RBF = ", mae_train_rbf)
print("Epsilon du test set RBF = ", mae_test_rbf)
print("Epsilon du training set Matern = ", mae_train_matern)
print("Epsilon du test set Matern = ", mae_test_matern)


Epsilon du training set RBF =  6.951473292575834e-09
Epsilon du test set RBF =  7.110916660845944
Epsilon du training set Matern =  3.743857522079499e-09
Epsilon du test set Matern =  4.5458169551879175


Comment here: 

# Boiler

![furnace_plotL](boiler/furnace_plotL.PNG)

This dataset contains high-frequency time-series  data collected (every 5 seconds) from an industrial boiler operating in a chemical plant. The boiler is equipped with multiple sensors capturing parameters such as pressure, temperature, flow rate, and oxygen levels. The dataset reflects a real-world industrial scenario. The boiler outlet steam temperature, ranging typically from 530 °C to 545 °C during stable operation, is used as the key indicator of equipment state. Deviations outside this range represent abnormal operating conditions. 

## 4. Poisson Process

4. During stable operations, the outlet steam temperature is in the interval 530 °C to 545 °C. 

a) Load the dataset 'data_boiler.csv', plot the Boiler outlet steam temperature (variable 'TE_8332A.AV_0') and count the number of times this temperature is outside this normal range. What do you observe? (**1 pt**)

In [12]:
#code here


Comment here: 

b) A Poisson process, denoted by $N_t$ is a counting process. The number of events observed during an interval [0,t] is distributed according to a Poisson law with a parameter $\lambda \times t$. Using the method of moment, estimate $\lambda \times t$, the frequency of overheating **or** underheating (i.e. when we are outside the interval) per hour.

Remark: do not forget that time-series data are collected every 5 seconds.

(**1.5 pt**)

In [13]:
#code here


c) Calculate the probability of observing more ( >= ) than 50 abnormal temperatures on 1h. (**1 pt**)

In [14]:
#code here


Comment here: 

## 5. Fit of distributions and forecasting 

5. The induced draft fan motor current must in normal condition stay below 30 Amp. A current above 30 Amp may cause damage to the installation. 

a) Fit a Gamma and an exponentiated Weibull distributions to the variable YFJ3_AI.AV_0. Compare histograms and  densities, and choose the most appropriate distribution. Using the most appropriate distribution, determine the probability that over a similar period of time, we observe a peak of induced draft fan motor current above 30 Amp. 
(**3 pts**)

In [15]:
#code here


Comment here: 

b) You want to set up a prediction algorithm of over- and under-heating (variable TE_8332A.AV_0). The aim is to anticipate any abnormal deviation to take necessary measures for driving back the temperature in $[530 ; 545]$. For this purpose, you will use the measure at time $ t - lag \times 5s$ for predicting the steam temperature at time t, where $lag$ is the number of 5-seconds lags. The model to fit is of the form:
$$Y_t = \beta_0 + \beta_1 X^1_{t-lag}+\beta_2 X^2_{t-lag}+...+\beta_n X^n_{t-lag}+\varepsilon,$$ 
where $Y$ is the target variable (i.e. TE_8332A.AV_0), $(X^1,...,X^n)$ are all the explanatory variables (i.e. all the variables except TE_8332A.AV_0) and $\varepsilon \sim N(0,1).$

* Create a dataset such that for each date $t$ (each line), you have the target variable at time $t$ and the explanatory variables at time $t-lag \times 5s$.
* Use the OLS() function of the package statsmodels.api to perform the linear regression. 
* If an explanatory variable is not significant, remove it from your model.
* Test different lags and determine  the maximum number of lags, such that the probabilities that your model detects over- and under-heatings are above 90%

(**4 pts**)

In [16]:
#code here


Comment here:

c)  Compare the probabilities that your model detects over- and under-heatings. (**1 pt**)

In [17]:
#code here


Comment here: 